# 02 — Индексация датасетов

Для каждого датасета из `EVAL_DATASETS`:
1. Создаём коллекцию `{prefix}_{dataset_name}` через CRUD API
2. Загружаем corpus в эту коллекцию
3. Smoke-test поиска

Все коллекции используют одну и ту же преднастройку из `config.py`.

**Важно:** коллекции создаются ОДИН раз и переиспользуются ноутбуками 03-05.

In [ ]:
import sys
import json
import time

sys.path.insert(0, "..")
from config import EVAL_COLLECTION_NAME_PREFIX, EVAL_DATASETS, DATA_DIR, PROJECT_ROOT

## 1. Создание коллекций

Каждая коллекция имеет имя `{prefix}_{dataset_name}` и использует одинаковую
преднастройку embedding + chunking.

In [ ]:
from evalkit.client import make_client
from evalkit.dataset_loaders import load_ir_dataset, load_hotpotqa, load_rubq_retrieval
from evalkit.csv_loader import clean_and_filter
import pandas as pd

client = make_client()
print(type(client).__name__)

In [ ]:
from evalkit.rag_config import base_config, collection_config, describe

collection_cfg = collection_config(base_config())
print(f"Config: {describe(collection_cfg)}")
collection_specs = []
print(f"Creating {len(EVAL_DATASETS)} collection(s)...")
for ci, (ds_key, ds_cfg) in enumerate(EVAL_DATASETS.items(), 1):
    coll_name = f"{EVAL_COLLECTION_NAME_PREFIX}_{ds_key}"
    t0 = time.perf_counter()
    resp = client.create_collection(
        collection_name=coll_name,
        description=f"Eval collection for {ds_key} ({ds_cfg['type']})",
        **collection_cfg,
    )
    elapsed = time.perf_counter() - t0
    print(
        f"  [{ci}/{len(EVAL_DATASETS)}] {coll_name} ({ds_key}): {resp.get('status', resp)} ({elapsed:.1f}s)"
    )
    collection_specs.append(
        {
            "dataset_key": ds_key,
            "collection_name": coll_name,
            "dataset_type": ds_cfg["type"],
        }
    )
collection_names = [cs["collection_name"] for cs in collection_specs]
print(f"\nCollections ready: {collection_names}")

## 2. Загрузка документов

Для каждого датасета загружаем corpus и индексируем в свою коллекцию.

In [ ]:
manifest_collections = []
for cs in collection_specs:
    ds_key = cs["dataset_key"]
    ds_cfg = EVAL_DATASETS[ds_key]
    coll_name = cs["collection_name"]
    ds_type = cs["dataset_type"]
    print(f"\n=== [{ds_key}] {ds_type} ===")
    print(f"Collection: {coll_name}")

    # Load corpus based on type
    if ds_type == "synthetic":
        print("Loading corpus from BPMSoft CSV...")
        df = pd.read_csv(
            str(PROJECT_ROOT / "data_bpmsoft" / "1.9-all-comma.csv"), sep=","
        )
        df = clean_and_filter(df)
        corpus = dict(zip(df["xml_id"], df["clean_text"]))
    elif ds_type == "beir":
        print(f"Loading BEIR: {ds_key}...")
        corpus, _ = load_ir_dataset(hf_name=ds_cfg["hf_name"])
    elif ds_type == "hotpotqa":
        print(f"Loading HotpotQA: {ds_key}...")
        corpus, _ = load_hotpotqa(
            hf_name=ds_cfg["hf_name"], max_examples=ds_cfg.get("max_examples")
        )
    elif ds_type == "rubq":
        print(f"Loading RuBQ Retrieval: {ds_key}...")
        corpus, _ = load_rubq_retrieval(hf_name=ds_cfg["hf_name"])
    else:
        raise ValueError(f"Unknown dataset type: {ds_type}")

    print(f"Corpus: {len(corpus)} documents")

    # Upload to collection (parallel)
    docs_list = list(corpus.items())
    docs_batch = [(doc_id, text, coll_name) for doc_id, text in docs_list]
    t0 = time.perf_counter()
    upload_results = client.upload_many(docs_batch, max_workers=16, desc=coll_name)
    elapsed = time.perf_counter() - t0
    # Collect task_ids for later polling
    task_ids = [tid for _, tid in upload_results if tid is not None]
    print(
        f"  Upload time: {elapsed:.1f}s ({len(corpus) / max(elapsed, 0.01):.1f} docs/s)"
    )
    manifest_collections.append(
        {
            "dataset_key": ds_key,
            "collection_name": coll_name,
            "dataset_type": ds_type,
            "doc_count": len(corpus),
            "upload_time_s": round(elapsed, 1),
            "task_ids": task_ids,
        }
    )

# Check all tasks completed (ожидание задач индексации agent-core)
print("\n=== Checking all tasks ===")
from tqdm import tqdm

total_failed = 0
for mc in manifest_collections:
    coll_name = mc["collection_name"]
    pending = list(mc["task_ids"])
    failed_tasks = []
    pbar = tqdm(total=len(pending), desc=f"indexing {coll_name}", unit="doc")
    while pending:
        still_pending = []
        for task_id, status in zip(pending, client.check_tasks(pending)):
            state = status.get("state", "unknown")
            if state == "completed":
                pbar.update(1)
            elif state == "failed":
                failed_tasks.append(task_id)
                pbar.set_postfix(failed=len(failed_tasks))
                pbar.update(1)
            else:
                still_pending.append(task_id)
        pending = still_pending
        if pending:
            time.sleep(1)
    pbar.close()
    if failed_tasks:
        print(f"  {len(failed_tasks)} tasks failed")
        total_failed += len(failed_tasks)
    else:
        print(f"  [OK] All {len(mc['task_ids'])} tasks completed")

if total_failed == 0:
    print("\nAll tasks completed!")
else:
    print(f"\nFailed: {total_failed} tasks")

## 3. Проверка: быстрый search

In [ ]:
print("Smoke test: search across all collections")
for coll_name in collection_names:
    t0 = time.perf_counter()
    results = client.search(
        "как создать новый документ", collection_name=coll_name, top_k=3
    )
    elapsed = time.perf_counter() - t0
    print(f"  {coll_name}: {len(results)} results ({elapsed:.2f}s)")
    for r in results[:2]:
        print(f"    score={r['score']:.3f}  doc={r['doc_id']}")

## 4. Сохранение manifest

In [ ]:
embedding_info = collection_cfg["embedding"]
chunking_info = collection_cfg["chunking"]
reranker_info = collection_cfg["reranker"]
DATA_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = DATA_DIR / "eval_manifest.json"
manifest = {
    "backend": "agent-core",
    "prefix": EVAL_COLLECTION_NAME_PREFIX,
    "embedding": embedding_info,
    "chunking": chunking_info,
    "reranker": reranker_info,
    "datasets": [cs["dataset_key"] for cs in manifest_collections],
    "collections": [cs["collection_name"] for cs in manifest_collections],
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "collection_details": manifest_collections,
}
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f"Manifest saved -> {manifest_path}")
print(json.dumps(manifest, ensure_ascii=False, indent=2))